In [1]:
import os 
import requests
url= r"https://arxiv.org/pdf/2412.19437"
pdf_path=os.path.join(os.getcwd() , "Data_file.pdf")

if not os.path.exists(pdf_path):

    response= requests.get(url)
    if response.status_code == 200:
        with open(pdf_path,"wb") as f:
            f.write(response.content)
        print(f"[INFO] the file is downloaded at : {pdf_path}")
else:
    print(f"[INFO] file exist : {pdf_path} ")

[INFO] file exist : e:\Tests\Data_file.pdf 


In [2]:
# import fitz  # PyMuPDF
# pages_and_texts=[]
# doc = fitz.open("Data_file.pdf")
# data_text = []

# for page_num , page in enumerate(doc):
#     text = page.get_text()
#     pages_and_texts.append({
#         "text_of_page" : text,
#         "page_number" : page_num + 1
#     })
# print(pages_and_texts[3]) 
# for text in pages_and_texts:
#     text["text_of_page"] =text["text_of_page"].replace(r"-\n"," ").strip()
# print(pages_and_texts[3] )

In [3]:
import fitz  # PyMuPDF

def clean_text(text: str):
    text = text.replace("\n", " ").strip()
    return text

def open_pdf(file_path:str):
    doc=fitz.open(file_path)
    pages_and_texts=[]
    for page_num , page in enumerate(doc):
        text = page.get_text()
        cleaned_text = clean_text(text)
        pages_and_texts.append({
            "text_of_page" : cleaned_text,
            "page_count_char" : len(cleaned_text),
            "page_count_word" : len(cleaned_text.split(" ")),
            "page_count_sentences" : len(cleaned_text.split(". ")),
            "tokens_num" : len(cleaned_text) // 4 , # i dont get it but it is what it is 
            "page_number" : page_num + 1
        })
    return pages_and_texts

pages_and_texts=open_pdf("Data_file.pdf")
pages_and_texts[3]

{'text_of_page': '1. Introduction In recent years, Large Language Models (LLMs) have been undergoing rapid iteration and evolution (Anthropic, 2024; Google, 2024; OpenAI, 2024a), progressively diminishing the gap to- wards Artificial General Intelligence (AGI). Beyond closed-source models, open-source models, including DeepSeek series (DeepSeek-AI, 2024a,b,c; Guo et al., 2024), LLaMA series (AI@Meta, 2024a,b; Touvron et al., 2023a,b), Qwen series (Qwen, 2023, 2024a,b), and Mistral series (Jiang et al., 2023; Mistral, 2024), are also making significant strides, endeavoring to close the gap with their closed-source counterparts. To further push the boundaries of open-source model capa- bilities, we scale up our models and introduce DeepSeek-V3, a large Mixture-of-Experts (MoE) model with 671B parameters, of which 37B are activated for each token. With a forward-looking perspective, we consistently strive for strong model performance and economical costs. Therefore, in terms of architectu

In [4]:
import pandas as pd 

df = pd.DataFrame(pages_and_texts)
df.head(10)

,text_of_page,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
0,DeepSeek-V3 Technical Report DeepSeek-AI resea...,1817,238,11,454,1
1,Contents 1 Introduction 4 2 Architecture 6 2.1...,2634,919,766,658,2
2,4.5.3 Batch-Wise Load Balance VS. Sequence-Wis...,1715,572,462,428,3
3,"1. Introduction In recent years, Large Languag...",4248,594,26,1062,4
4,Training Costs Pre-Training Context Extension ...,3287,468,21,821,5
5,verification and reflection patterns of R1 int...,3475,457,23,868,6
6,… Router Input Hidden 𝐮𝐮𝑡𝑡 Output Hidden 𝐡𝐡𝑡𝑡 ...,1557,262,8,389,7
7,where c𝐾𝑉 𝑡 ∈R𝑑𝑐is the compressed latent vecto...,2236,353,9,559,8
8,where 𝑁𝑠and 𝑁𝑟denote the numbers of shared exp...,2901,455,17,725,9
9,Main Model (Next Token Prediction) Embedding L...,2904,451,23,726,10


In [5]:
df.describe().round(2)

,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
count,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00
std,799.86,178.18,124.63,199.99,15.44
min,699.00,99.00,1.00,174.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00
50%,2992.00,457.00,23.00,748.00,27.00
75%,3281.00,516.00,29.00,820.00,40.00
max,4248.00,919.00,766.00,1062.00,53.00


# Turn text to sentences 

In [6]:
from spacy.lang.en import English

nlp= English()

# add sentencizer to pipeliine

nlp.add_pipe("sentencizer")

#example sentence to test
semtemces = "this my name . hellow mr.ahmed . what is your name ?"

doc = nlp(semtemces)
# list the sentences to show 
list(doc.sents)


[this my name ., hellow mr.ahmed ., what is your name ?]

# Now try the `data` 

In [7]:
for page in pages_and_texts:
    page["sentences"] = list(nlp(page["text_of_page"]).sents)
    page["sentences_num"] =len(page["sentences"])

#make sure tha all sentences is string not spacy type
for page in pages_and_texts:
    page["sentences"]=[str(sentences) for sentences in page["sentences"]]

pages_and_texts[5]

{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [8]:
#read specific sentence
pages_and_texts[5]["sentences"][6]

'Code, Math, and Reasoning: (1) DeepSeek-V3 achieves state-of-the-art performance on math-related benchmarks among all non-long-CoT open-source and closed-source models.'

In [9]:
# see some statistics about this data 
df =pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num
count,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94
std,799.86,178.18,124.63,199.99,15.44,14.44
min,699.00,99.00,1.00,174.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00


In [10]:
test=list(range(1,25))
input_list=[]
slice_size=10
x=[test[i:i+slice_size] for i in range(0,len(test),slice_size)]
x

[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
 [21, 22, 23, 24]]

In [11]:
def chunker (input_list : list[str],slice_size:int)->list[list[str]]:

    return[input_list[i:i+slice_size] for i in range(0,len(input_list),slice_size)]

for page in pages_and_texts:
    page["sentences_chunk"] = chunker(page["sentences"],slice_size)
    page["sentences_chunk_num"] =len( page["sentences_chunk"]) 

pages_and_texts[5]

{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [12]:
df =pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num,sentences_chunk_num
count,53.00,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94,2.89
std,799.86,178.18,124.63,199.99,15.44,14.44,1.40
min,699.00,99.00,1.00,174.00,1.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00,2.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00,3.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00,3.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00,6.00


In [13]:
pages_and_chunk=[]
for page in pages_and_texts:
    for chunk in page["sentences_chunk"]:
        chunk_dict = {}

        # join all sentences in chunk to be one paragraphe
        paragragh = "".join(chunk).replace("  "," ").strip()
        chunk_dict ["paragraphe"] = paragragh
        
        # Calculate statistics
        words = paragragh.split(" ")
        word_count = len(words)
        char_count = len(paragragh)
        tokens_count = char_count // 4  # approximate tokens
        sentence_count = len(chunk)  # number of sentences in chunk
        
        # Store statistics
        chunk_dict["word_count"] = word_count
        chunk_dict["char_count"] = char_count
        chunk_dict["tokens_count"] = tokens_count
        chunk_dict["sentence_count"] = sentence_count
        chunk_dict ["page_number"] = page["page_number"]

        pages_and_chunk.append(chunk_dict)
pages_and_chunk[5] ,len(pages_and_chunk),len(pages_and_texts)

({'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
  'word_count': 298,
  'char_count': 830,
  'tokens_count': 207,
  'sentence_count': 10,
  'page_number': 3},
 153,
 53)

In [14]:
df = pd.DataFrame(pages_and_chunk)
df.describe().round(2)

,word_count,char_count,tokens_count,sentence_count,page_number
count,153.00,153.00,153.00,153.00,153.00
mean,157.03,974.48,243.26,8.29,26.52
std,145.88,584.64,146.14,2.95,13.86
min,1.00,1.00,0.00,1.00,1.00
25%,74.00,572.00,143.00,7.00,15.00
50%,134.00,912.00,228.00,10.00,28.00
75%,191.00,1301.00,325.00,10.00,39.00
max,877.00,2975.00,743.00,10.00,53.00


In [15]:
df[df["word_count"]<4]
pages_and_texts[8]

{'text_of_page': 'where 𝑁𝑠and 𝑁𝑟denote the numbers of shared experts and routed experts, respectively; FFN(𝑠) 𝑖 (·) and FFN(𝑟) 𝑖 (·) denote the 𝑖-th shared expert and the 𝑖-th routed expert, respectively; 𝐾𝑟denotes the number of activated routed experts; 𝑔𝑖,𝑡is the gating value for the 𝑖-th expert; 𝑠𝑖,𝑡is the token-to-expert affinity; e𝑖is the centroid vector of the 𝑖-th routed expert; and Topk(·, 𝐾) denotes the set comprising 𝐾highest scores among the affinity scores calculated for the 𝑡-th token and all routed experts. Slightly different from DeepSeek-V2, DeepSeek-V3 uses the sigmoid function to compute the affinity scores, and applies a normalization among all selected affinity scores to produce the gating values. Auxiliary-Loss-Free Load Balancing. For MoE models, an unbalanced expert load will lead to routing collapse (Shazeer et al., 2017) and diminish computational efficiency in scenarios with expert parallelism. Conventional solutions usually rely on the auxiliary loss (Fedus e

In [16]:
#filter out tokens less than 30
filtered_chunks = [chunk for chunk in pages_and_chunk if chunk["tokens_count"] >= 30]
filtered_chunks[111]

{'paragraphe': 'A study of bfloat16 for deep learning training.arXiv preprint arXiv:1905.12322, 2019.S. Krishna, K. Krishna, A. Mohananey, S. Schwarcz, A. Stambler, S. Upadhyay, and M. Faruqui.Fact, fetch, and reason: A unified evaluation of retrieval-augmented generation.CoRR, abs/2409.12941, 2024.doi: 10.48550/ARXIV.2409.12941.URL https://doi.org/10.485 50/arXiv.2409.12941.T. Kwiatkowski, J. Palomaki, O. Redfield, M. Collins, A. P. Parikh, C. Alberti, D. Epstein, I. Polosukhin, J. Devlin, K. Lee, K. Toutanova, L. Jones, M. Kelcey, M. Chang, A. M. Dai, J. Uszkoreit, Q. Le, and S. Petrov.Natural questions: a benchmark for question answering research.Trans.',
 'word_count': 84,
 'char_count': 648,
 'tokens_count': 162,
 'sentence_count': 10,
 'page_number': 39}

In [17]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [19]:
set = ["my name", "her name", "the naem"]
embeding = model.encode(set)
embeding = dict(zip(set,embeding))

embeding,embeding["my name"].shape

({'my name': array([-1.11268625e-01,  3.23447920e-02, -3.83406430e-02,  2.23897584e-02,
         -4.03141081e-02, -5.62008880e-02,  2.33329892e-01,  3.92086953e-02,
         -1.01079056e-02, -6.38170093e-02,  1.64090171e-02, -3.45565043e-02,
          4.02090475e-02, -7.56014735e-02, -8.34893435e-03,  2.65754201e-02,
         -1.42914457e-02,  6.17227843e-03, -8.46135393e-02, -7.72886723e-02,
         -4.17722203e-02,  6.27690181e-02, -3.24106877e-05, -2.02984530e-02,
         -8.92573446e-02,  6.10290328e-03, -3.12836133e-02,  9.11155716e-02,
         -3.69638987e-02, -1.61543280e-01, -3.80763202e-04,  7.48014171e-03,
          9.98783484e-02,  5.72649240e-02, -4.66819294e-03, -4.17357087e-02,
         -1.26731232e-01, -2.11647116e-02,  5.62063605e-02, -1.07810339e-02,
         -4.70564887e-03, -1.19292378e-01,  2.42646076e-02,  5.37947472e-03,
          7.11646024e-03,  7.88040832e-03,  6.28919399e-04,  2.45067608e-02,
          1.61230154e-02,  9.47958678e-02, -9.59325731e-02, -8.88

In [20]:
from tqdm import tqdm

for chunk in tqdm(pages_and_chunk):
    chunk["embeding"] = model.encode(chunk["paragraphe"])


100%|██████████| 153/153 [00:07<00:00, 20.06it/s]


In [21]:
import random

random.sample(pages_and_chunk,k=1)

[{'paragraphe': '… Router Input Hidden 𝐮𝐮𝑡𝑡 Output Hidden 𝐡𝐡𝑡𝑡 ′ 1 𝑁𝑁𝑠𝑠 1 2 𝑁𝑁𝑟𝑟-1 𝑁𝑁𝑟𝑟 Shared Expert Routed Expert Top-𝐾𝐾𝑟𝑟 Attention Feed-Forward Network … 3 4 RMSNorm RMSNorm Transformer Block ×𝐿𝐿 DeepSeekMoE 0 Input Hidden 𝐡𝐡𝑡𝑡 Multi-Head Latent Attention (MLA) 0 {𝐪𝐪𝑡𝑡,𝑖𝑖 𝐶𝐶} {𝐯𝐯𝑡𝑡,𝑖𝑖 𝐶𝐶} {𝐤𝐤𝑡𝑡,𝑖𝑖 𝐶𝐶} Latent 𝐜𝐜𝑡𝑡 𝐾𝐾𝐾𝐾 Latent 𝐜𝐜𝑡𝑡 𝑄𝑄 {𝐪𝐪𝑡𝑡,𝑖𝑖 𝑅𝑅} 𝐤𝐤𝑡𝑡 𝑅𝑅 Cached During Inference Multi-Head Attention concatenate concatenate {[𝐪𝐪𝑡𝑡,𝑖𝑖 𝐶𝐶; 𝐪𝐪𝑡𝑡,𝑖𝑖 𝑅𝑅]} {[𝐤𝐤𝑡𝑡,𝑖𝑖 𝐶𝐶; 𝐤𝐤𝑡𝑡 𝑅𝑅]} … Output Hidden 𝐮𝐮𝑡𝑡 … … … … … 1 … … … … apply RoPE apply RoPE Figure 2 | Illustration of the basic architecture of DeepSeek-V3.Following DeepSeek-V2, we adopt MLA and DeepSeekMoE for efficient inference and economical training.strategy (Wang et al.,2024a) for DeepSeekMoE to mitigate the performance degradation induced by the effort to ensure load balance.Figure 2 illustrates the basic architecture of DeepSeek-V3, and we will briefly review the details of MLA and DeepSeekMoE in this section.2.1.1.Multi-Head Latent 

In [22]:
page_and_Embedding_df = pd.DataFrame(pages_and_chunk)
page_and_Embedding_file_name="page_and_Embedding.csv"
page_and_Embedding_path = os.path.join(os.getcwd(),page_and_Embedding_file_name)
page_and_Embedding_df.to_csv(page_and_Embedding_path,index=False)

In [23]:
pages_and_chunk[5]

{'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 'word_count': 298,
 'char_count': 830,
 'tokens_count': 207,
 'sentence_count': 10,
 'page_number': 3,
 'embeding': array([-5.76343089e-02,  1.8627015

In [24]:
pages_and_chunk=[]
pages_and_chunk

[]

In [29]:
pages_and_chunk =pd.read_csv(page_and_Embedding_path)
data = pages_and_chunk.to_dict(orient="records")
data[5]

{'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 'word_count': 298,
 'char_count': 830,
 'tokens_count': 207,
 'sentence_count': 10,
 'page_number': 3,
 'embeding': '[-5.76343089e-02  1.86270159e-02 